# CatSense – Model Preparation & Dataset Visualization

This notebook:
1. **Checks** if trained weights exist in `models/weights/`.
2. **Trains** the model from scratch on `expert_collection_1` + `expert_collection_2` if weights are missing.
3. **Generates** the *Sample Waveform & Mel-Spectrogram per Class* figure.
4. **Saves** all artefacts to `research/model_preparation/`.

In [1]:
import sys, os
# Make server/src importable
sys.path.insert(0, os.path.abspath('../../server'))

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import cv2
import noisereduce as nr
import pickle

from pathlib import Path
from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import to_categorical

# Import AttentionLayer so Keras can deserialize it
import src.config as config
from src.models import AttentionLayer  # registers the custom layer

# ── Config constants ──────────────────────────────────────────────────────────
SR         = config.SR
DURATION   = config.DURATION
IMG_SIZE   = config.IMG_SIZE
BATCH_SIZE = config.BATCH_SIZE
CATEGORIES = config.CATEGORIES

CAT_DB_ROOT      = Path(config.EXPERT_POOL_1)
CAT_SAMPLES_ROOT = Path(config.EXPERT_POOL_2)
MODEL_VAULT      = Path(config.MODEL_VAULT)

OUTPUT_DIR = Path(__file__).resolve().parent if '__file__' in dir() else Path('.')
# When running via nbconvert the cwd is the project root; adjust accordingly
OUTPUT_DIR = Path('research/model_preparation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_VAULT / 'catsense_final_best.keras'
PKL_PATH   = MODEL_VAULT / 'catsense_final_best.pkl'

print(f'TensorFlow : {tf.__version__}')
print(f'Model vault: {MODEL_VAULT}')
print(f'Output dir : {OUTPUT_DIR.resolve()}')

TensorFlow : 2.21.0
Model vault: /Users/ajay/project/models/weights
Output dir : /Users/ajay/project/research/model_preparation/research/model_preparation


## 1 – Model Check / Train

We load the model if weights already exist; otherwise we re-train from scratch.

In [2]:
def get_audio_files(root_dir):
    """Walk a structured directory and collect (path, label) pairs."""
    files = []
    for cat in CATEGORIES:
        p = root_dir / cat
        if p.exists():
            for ext in ['*.mp3', '*.wav']:
                for f in p.glob(ext):
                    files.append({'path': str(f), 'label': cat})
    return files


def preprocess_audio(path):
    """Load → denoise → normalize → pad/trim → mel + mfcc → 3-channel tensor."""
    y, _ = librosa.load(path, sr=SR)
    y    = nr.reduce_noise(y=y, sr=SR, prop_decrease=0.7)
    y    = librosa.util.normalize(y)
    t_len = int(SR * DURATION)
    y    = np.pad(y, (0, max(0, t_len - len(y))), mode='constant')[:t_len]

    mel  = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128)
    mel  = cv2.resize(librosa.power_to_db(mel), IMG_SIZE)
    mfcc = cv2.resize(librosa.feature.mfcc(y=y, sr=SR, n_mfcc=40), IMG_SIZE)

    def norm(a):
        return (a - a.min()) / (a.max() - a.min() + 1e-8)

    return np.stack([norm(mel), norm(mel), norm(mfcc)], axis=-1)


def build_prod_model():
    """Simple MobileNetV2 + Dense head (no AttentionLayer)."""
    base = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base.trainable = True
    x   = layers.GlobalAveragePooling2D()(base.output)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(len(CATEGORIES), activation='softmax')(x)
    m   = Model(base.input, out)
    m.compile(
        optimizer=keras.optimizers.Adam(config.LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return m

In [3]:
le = LabelEncoder()
le.fit(CATEGORIES)
model = None

if MODEL_PATH.exists():
    print(' Model weights found – loading…')
    try:
        model = keras.models.load_model(
            str(MODEL_PATH),
            custom_objects={'AttentionLayer': AttentionLayer}
        )
        if PKL_PATH.exists():
            with open(str(PKL_PATH), 'rb') as f:
                le = pickle.load(f)
        print(f'   Loaded: {MODEL_PATH.name}  ✔')
    except Exception as e:
        print(f'   Could not load existing model ({e}). Will re-train.')
        model = None

if model is None:
    print(' Training from scratch…')

    files = get_audio_files(CAT_DB_ROOT) + get_audio_files(CAT_SAMPLES_ROOT)
    df    = pd.DataFrame(files)
    print(f'   Expert samples: {len(df)}')

    X, Y = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Preprocessing'):
        try:
            feat = preprocess_audio(row['path'])
            X.append(feat)
            Y.append(row['label'])
            X.append(feat + np.random.normal(0, 0.01, feat.shape))
            Y.append(row['label'])
        except Exception as e:
            print(f'   Skip {row["path"]}: {e}')

    X = np.array(X, dtype=np.float32)
    le.fit(Y)
    Y_enc = to_categorical(le.transform(Y))

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, Y_enc, test_size=0.15, stratify=le.transform(Y), random_state=42
    )

    model = build_prod_model()
    model.fit(
        X_tr, y_tr,
        epochs=config.EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        verbose=1
    )

    y_pred = model.predict(X_te).argmax(axis=1)
    y_true = y_te.argmax(axis=1)
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=le.classes_))

    MODEL_VAULT.mkdir(parents=True, exist_ok=True)
    new_model_path = MODEL_VAULT / 'catsense_model_prep.keras'
    new_pkl_path   = MODEL_VAULT / 'catsense_model_prep.pkl'
    model.save(str(new_model_path))
    with open(str(new_pkl_path), 'wb') as f:
        pickle.dump(le, f)

    import shutil
    shutil.copy(new_model_path, OUTPUT_DIR / new_model_path.name)
    shutil.copy(new_pkl_path,   OUTPUT_DIR / new_pkl_path.name)
    print(f'   Saved new weights → {new_model_path}')

print('\nModel ready.')

 Model weights found – loading…
   Loaded: catsense_final_best.keras  ✔

Model ready.


## 2 – Sample Waveform & Mel-Spectrogram per Class

For each of the 10 emotion categories one representative audio file is visualised:
* **Left column** – raw waveform
* **Right column** – Mel-spectrogram (dB, magma colormap)

In [4]:
def pick_sample(category):
    """Return the path of the first available audio file for a category."""
    for root in [CAT_DB_ROOT, CAT_SAMPLES_ROOT]:
        p = root / category
        if p.exists():
            for ext in ['*.mp3', '*.wav']:
                matches = sorted(p.glob(ext))
                if matches:
                    return matches[0]
    return None


n_cats = len(CATEGORIES)
fig, axes = plt.subplots(n_cats, 2, figsize=(14, n_cats * 2.6))
fig.suptitle(
    'Sample Waveform & Mel-Spectrogram per Class',
    fontsize=14, fontweight='bold', y=1.002
)

for row_idx, cat in enumerate(CATEGORIES):
    ax_wave = axes[row_idx, 0]
    ax_spec = axes[row_idx, 1]

    sample_path = pick_sample(cat)
    if sample_path is None:
        ax_wave.text(0.5, 0.5, 'No sample', ha='center', va='center')
        ax_spec.text(0.5, 0.5, 'No sample', ha='center', va='center')
        continue

    y_raw, sr = librosa.load(str(sample_path), sr=SR)

    # ── Waveform ─────────────────────────────────────────────────────────────
    t = np.linspace(0, len(y_raw) / sr, num=len(y_raw))
    ax_wave.plot(t, y_raw, color='steelblue', linewidth=0.5)
    ax_wave.set_title(f'{cat} — Waveform', fontsize=9)
    ax_wave.set_xlabel('Time', fontsize=7)
    ax_wave.set_ylabel('Amplitude', fontsize=7)
    ax_wave.tick_params(labelsize=6)
    ax_wave.set_xlim([0, t[-1]])
    ax_wave.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

    # ── Mel-Spectrogram ───────────────────────────────────────────────────────
    mel    = librosa.feature.melspectrogram(y=y_raw, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img    = librosa.display.specshow(
        mel_db, sr=sr, x_axis='time', y_axis='mel',
        ax=ax_spec, cmap='magma'
    )
    ax_spec.set_title(f'{cat} — Mel Spectrogram', fontsize=9)
    ax_spec.set_xlabel('Time', fontsize=7)
    ax_spec.set_ylabel('Hz', fontsize=7)
    ax_spec.tick_params(labelsize=6)
    cbar = fig.colorbar(img, ax=ax_spec, format='%+2.0f dB')
    cbar.ax.tick_params(labelsize=6)

plt.tight_layout()

out_png = OUTPUT_DIR / 'sample_waveforms_melspecs.png'
plt.savefig(str(out_png), dpi=120, bbox_inches='tight')
plt.show()
print(f' Plot saved → {out_png.resolve()}')

 Plot saved → /Users/ajay/project/research/model_preparation/research/model_preparation/sample_waveforms_melspecs.png


/var/folders/s7/yc1pvgjd40v4n0v3bj6kpp7w0000gn/T/ipykernel_18244/1445649191.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3 – Summary of Artefacts

In [5]:
print('Artefacts saved to:', OUTPUT_DIR.resolve())
print()
for p in sorted(OUTPUT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<50}  {size_kb:>10.1f} KB')

Artefacts saved to: /Users/ajay/project/research/model_preparation/research/model_preparation

  sample_waveforms_melspecs.png                            988.1 KB
